# Chat avec le Qwen3-14B obfusqué — accès via Tailscale

Le modèle obfusqué `qwen3-14b-h128-a1-h02` est servi **serverless sur Modal**
(A100-40GB). Un **proxy OpenAI-compatible** tourne sur `sanroque` (service
systemd `obfuscator-proxy`, clé de permutation locale), exposé sur le
**tailnet** par `tailscale serve` :

```
https://sanroque.tailc69141.ts.net:9443/v1   ← endpoint OpenAI (tailnet only)
```

Ce notebook fonctionne **depuis n'importe quelle machine du tailnet** (buho,
un autre poste…) — il suffit d'avoir Tailscale connecté. Le trafic est
chiffré par le tailnet ; la clé Π ne quitte jamais sanroque.

> Prérequis réseau : `tailscale up` sur la machine qui exécute ce notebook.

In [ ]:
import requests

# Endpoint OpenAI-compatible du proxy (via Tailscale, HTTPS magique)
BASE_URL = "https://sanroque.tailc69141.ts.net:9443/v1"
MODEL = "qwen3-14b-h128-a1-h02"

# 1. Vérifier la connexion (health + modèles disponibles)
r = requests.get(BASE_URL.replace("/v1", "") + "/health", timeout=30)
print("health:", r.json())
r = requests.get(BASE_URL + "/models", timeout=30)
print("modèles:", [m["id"] for m in r.json()["data"]])

## Envoyer un prompt

La fonction `chat()` envoie des messages au format OpenAI
(`/v1/chat/completions`) et renvoie la réponse du modèle.

In [ ]:
def chat(messages, max_tokens=200, temperature=None):
    """Appelle le modèle obfusqué via le proxy Tailscale (format OpenAI)."""
    payload = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if temperature is not None:
        payload["temperature"] = temperature   # accepté mais ignoré (greedy)
    r = requests.post(BASE_URL + "/chat/completions",
                      json=payload, timeout=600)
    r.raise_for_status()
    d = r.json()
    return (d["choices"][0]["message"]["content"],
            d["usage"]["prompt_tokens"], d["usage"]["completion_tokens"])

# --- EXEMPLE 1 : question simple ---
reponse, pt, ct = chat([
    {"role": "user", "content": "Quelle est la capitale de la France ? "
                                "Réponds en un mot."},
], max_tokens=50)
print("Q : Quelle est la capitale de la France ?")
print("R :", reponse)
print(f"   ({pt} tokens prompt, {ct} tokens réponse)")

## Exemples d'usage

- **Résumé** : le modèle résume un texte en français (format wiki possible).
- **Multi-tour** : l'historique est renvoyé à chaque appel.

In [ ]:
# --- EXEMPLE 2 : résumé court ---
texte = ("La confidentialité des échanges avec un LLM hébergé dans le cloud "
         "est un enjeu majeur pour les professions réglementées. Une approche "
         "consiste à transformer à la fois les données et le modèle, de sorte "
         "que le serveur ne manipule que des jetons permutés, illisibles sans "
         "la clé de permutation détenue par le client.")
reponse, _, _ = chat([
    {"role": "user",
     "content": f"Résume ce texte en 2 phrases : {texte}"},
], max_tokens=120)
print("Résumé :", reponse)

In [ ]:
# --- EXEMPLE 3 : conversation multi-tour ---
messages = [
    {"role": "user", "content": "Quelle est la capitale de la Belgique ?"},
    {"role": "assistant", "content": "Bruxelles."},
    {"role": "user", "content": "Et la capitale de la Suisse ? Réponds en un mot."},
]
reponse, _, _ = chat(messages, max_tokens=50)
print("R :", reponse)

## Chat interactif (à exécuter manuellement)

Exécutez la cellule ci-dessous pour une petite boucle de conversation dans le
notebook. Tapez `quit` pour sortir.

> Ne pas inclure cette cellule dans une exécution automatique (`nbconvert`)
> : elle attend une saisie clavier.

In [ ]:
def chat_interactif():
    messages = []
    print("Chat avec", MODEL, "(tape 'quit' pour sortir)\n")
    while True:
        user = input("Vous : ")
        if user.strip().lower() in ("quit", "exit"):
            break
        messages.append({"role": "user", "content": user})
        reponse, _, _ = chat(messages, max_tokens=250)
        print("Modèle :", reponse)
        messages.append({"role": "assistant", "content": reponse})

# chat_interactif()   ← décommentez pour lancer

## Notes

- **Modèle** : Qwen3-14B obfusqué (h>0, α_e=1,0/α_h=0,2) — défense mesurée :
  VMA gate 10,5 % / W_e·W_h 0 % / combiné 7,05 % ; qualité Q&A −1,1 % vs base.
- **Sécurité** : la clé de permutation reste sur sanroque (proxy local) ; le
  tailnet chiffre le transport ; Modal ne voit que des ids permutés.
- **Coût** : serverless A100-40GB, facturé au temps GPU réellement allumé
  (scale-to-zero) + cold start après inactivité (~1-2 min).
- **Limites** : pas de streaming (réponse complète), greedy (temperature
  ignorée), le proxy est mono-utilisateur sur le tailnet.
- Gestion du service sur sanroque :
  `systemctl --user status obfuscator-proxy` (logs : `journalctl --user -u
  obfuscator-proxy -f`).